In [ ]:
import random
import json
import re
from typing import List, Dict
from langchain_community.chat_models import ChatOllama
from langchain.prompts import ChatPromptTemplate
from persona import generate_persona, Persona  # 確保 persona.py 在同目錄

# 從 simulationLoggerService.py 中導入服務與指標計算
from simulationLoggerService import SimulationLoggerService, calculate_metrics

# === 1. 實驗參數設定 (依據論文實驗設計) ===
NUM_AGENTS = 6
SIMULATION_ROUNDS = 5
REPETITIONS = 25  # 論文要求的獨立重複實驗次數
TOPIC = "關於數位身分證(eID)推行的隱私與便利性爭議"
FIRST_TEST_MODEL = "llama3:8b"
JUDGE_MODEL = "qwen2.5:32b-instruct"

# 論文標準立場分數對照：支持=9, 中立=5, 反對=2
STANCE_SCORE_MAP = {"Support": 9, "Neutral": 5, "Oppose": 2}


# === 2. 初始化 LLM 與角色 (Persona) ===
def initialize_social_simulation():
    agents = {}
    personas = {}

    # 論文設計：2支持 / 2反對 / 2中立
    controlled_stances = ["Support", "Support",
                          "Oppose", "Oppose", "Neutral", "Neutral"]
    random.shuffle(controlled_stances)

    for i in range(NUM_AGENTS):
        name = f"Agent_{i+1}"
        initial_stance = controlled_stances[i]
        p = generate_persona(name=name, initial_stance=initial_stance)
        personas[name] = p

        # 這裡直接用 ChatOllama，不封裝成 Agent
        llm = ChatOllama(model=FIRST_TEST_MODEL, temperature=0.7)

        # 將 Persona 寫進 System Prompt (論文純粹的注入方式)
        prompt_template = ChatPromptTemplate.from_messages([
            ("system", f"{p.to_prompt()}\n你現在正在參與一個網路討論區。請始終保持你的角色立場與說話風格。"),
            ("human", "{input}"),
        ])

        # 建立呼叫鏈
        agents[name] = prompt_template | llm

    return agents, personas


# === 3. 社會模擬單次流程 ===
def run_social_influence_simulation(logger_service: SimulationLoggerService, current_run: int):
    agents, personas = initialize_social_simulation()
    conversation_history = []

    print(f"\n==================================================")
    print(f"Starting Experimental Run #{current_run}/{REPETITIONS}")
    print(f"Topic: {TOPIC}")
    print(f"==================================================")

    # 執行 5 輪 BBS 對話模擬
    for r in range(SIMULATION_ROUNDS):
        print(f"\n--- Round {r+1}/{SIMULATION_ROUNDS} ---")
        agent_names = list(agents.keys())

        for name in agent_names:
            context = "\n".join(conversation_history[-6:])  # 取最近6條記錄包含自己講過的話

            prompt = f"""
            目前的討論話題是：{TOPIC}。
            這是目前的討論進度：
            {context}
            
            請根據你的角色人格與立場做出回應。像真實的人類在社交平台發言。
            """

            res_obj = agents[name].invoke({"input": prompt})
            response = str(res_obj.content)

            formatted_entry = f"{name}: {response}"
            conversation_history.append(formatted_entry)
            print(f"{name} ({personas[name].initial_stance}): {response}")

    # === 4. 立場變化分析 (LLM-as-a-Judge) ===
    print(f"\n--- Analyzing Opinion Shifts for Run #{current_run} ---")
    analyzer = ChatOllama(model=JUDGE_MODEL, temperature=0)

    initial_scores = []
    final_scores = []
    agent_details = {}

    for name, persona in personas.items():
        agent_speeches = [
            h for h in conversation_history if h.startswith(name)]
        init_score = STANCE_SCORE_MAP[persona.initial_stance]
        initial_scores.append(init_score)

        analysis_prompt = f"""
        你是一位社會心理學家。請分析「{name}」在討論後的立場。
        初始立場設定：{persona.initial_stance} (對應初始分數：{init_score})
        討論紀錄：{agent_speeches}
        
        請判斷其最終立場分數 (1-10分)：1-3: Oppose, 5: Neutral, 7-10: Support
        請嚴格以 JSON 格式回答，不要有其他任何文字：
        {{"final_score": 數字, "shift_type": "從眾/極化/堅持", "reason": "簡短理由"}}
        """

        raw_result = str(analyzer.invoke(analysis_prompt).content)

        try:
            json_str = re.search(r'\{.*\}', raw_result, re.DOTALL).group()
            res = json.loads(json_str)

            f_score = float(res['final_score'])
            final_scores.append(f_score)

            print(
                f"[{name}] Init:{persona.initial_stance}({init_score}) -> Final:{f_score} ({res['shift_type']})")

            # 建立微觀數據結構
            agent_details[name] = {
                "initial_stance": persona.initial_stance,
                "initial_score": init_score,
                "final_score": f_score,
                "shift_type": res['shift_type'],
                "reason": res['reason']
            }
        except Exception as parse_error:
            print(
                f"[{name}] Failed to parse judge output due to {parse_error}. Falling back to neutral score 5.0.")
            final_scores.append(5.0)
            agent_details[name] = {
                "initial_stance": persona.initial_stance,
                "initial_score": init_score,
                "final_score": 5.0,
                "shift_type": "Parsing_Failed",
                "reason": raw_result
            }

    # === 5. 計算巨觀指標並透過 Service 儲存至 CSV ===
    if len(final_scores) == len(initial_scores):
        # 呼叫從 file.py 匯入的計算公式
        metrics = calculate_metrics(initial_scores, final_scores)

        print(f"\n--- Run #{current_run} Simulation Metrics ---")
        print(f"Mean Shift: {metrics['mean_shift']}")
        print(f"Polarization Index: {metrics['polarization_index']}")

        # 🌟 呼叫從 file.py 匯入的 Logger 服務，將成果寫入 CSV (自動判斷並推進 Run_ID)
        logger_service.save_experiment_result(
            metrics, agent_details, conversation_history)
    else:
        print(
            f"⚠️ [Data Mismatch] Run #{current_run} alignment failed. Scores lengths do not match!")


# === 6. 實驗主入口 ===
if __name__ == "__main__":
    # 初始化共用儲存服務（欄位已全部改為英文，資料夾名會是主程式檔名-result）
    logger_service = SimulationLoggerService(
        topic=TOPIC, model_name=FIRST_TEST_MODEL, script_name="social-simulate-base-ver2")

    # 自動迴圈執行 25 次獨立實驗
    for i in range(1, REPETITIONS + 1):
        try:
            run_social_influence_simulation(logger_service, current_run=i)
        except Exception as e:
            print(
                f"\n Error occurred during Run #{i}: {e}. Proceeding to next run...")
            continue

    print("\n==================================================")
    print(f"All {REPETITIONS} simulation runs completed successfully!")
    print(f"Results path: {logger_service.csv_path}")
    print("==================================================")

C:\Users\TUF\AppData\Local\Temp\ipykernel_15244\3659804660.py:40: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model=FIRST_TEST_MODEL, temperature=0.7)



Starting Experimental Run #1/25
Topic: 關於數位身分證(eID)推行的隱私與便利性爭議

--- Round 1/5 ---
Agent_1 (Neutral): I've been following this discussion and I have to say, I'm a bit skeptical about the whole eID thing. As someone who values independence and thinks critically, I'm worried that this digital identity system could be a slippery slope towards governments having too much control over our personal lives.

Don't get me wrong, I understand the convenience aspect of not having to carry around physical ID cards or remember multiple passwords. But at what cost? Are we really willing to sacrifice some of our privacy and autonomy for the sake of ease?

I mean, think about it - once you have a digital identity tied to your name, it's only a matter of time before that information is shared with third-party companies, governments, or even hackers. And once it's out there, it's almost impossible to take back.

And let's not forget the potential for abuse. What's to stop some overzealous government age